# Building a Vocabulary Tutor with Claude and the Kelly Intelligence API

In this cookbook we'll give Claude a `lookup_word` tool backed by [Kelly Intelligence](https://api.thedailylesson.com)'s public vocabulary API, then use Claude's tool-use loop to build a personalized vocabulary tutor that defines a word, gives translations, and quizzes the learner.

**Why Kelly?** Kelly Intelligence is a public-benefit vocabulary database with **162,253 English headwords**, **601K translations across 47 languages**, IPA pronunciations, etymologies, mnemonics, and a related-word graph. The `/v1/word/{word}` endpoint is **public and rate-limited per IP** (60 requests/hour), so this entire notebook only needs an Anthropic API key — no Kelly account required.

**What you'll learn:**

1. How to call the Kelly `/v1/word/{word}` endpoint directly with `requests`
2. How to wrap that call as a Claude tool with a clean input schema
3. How to run Claude's tool-use loop to answer learner questions about vocabulary
4. How to compose a multi-step tutor that defines, translates, and quizzes — all in one prompt


## Step 1: Set up the environment

Install dependencies and load your Anthropic API key. Kelly's endpoint needs no key.


In [1]:
%pip install -q anthropic requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os

import requests
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"

client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"
KELLY_URL = "https://api.thedailylesson.com/v1/word"

## Step 2: Call the Kelly endpoint directly

Before wiring Claude in, let's see what the raw API returns. We'll look up `ephemeral` and ask for translations into Spanish, French, German, and Japanese.


In [3]:
def kelly_lookup(word: str, translations: list[str] | None = None) -> dict:
    """Look up a word in the Kelly Intelligence vocabulary database.

    Args:
        word: The English headword to look up.
        translations: Optional list of ISO language codes (e.g., ["ES", "FR", "JA"]).
            Max 10. Defaults to ["ES", "FR", "DE"] on the server.

    Returns:
        A dict with definition, IPA, etymology, mnemonic, translations, and related words.
    """
    params = {}
    if translations:
        params["translations"] = ",".join(translations)
    response = requests.get(f"{KELLY_URL}/{word}", params=params, timeout=15)
    response.raise_for_status()
    return response.json()


# Try it
example = kelly_lookup("ephemeral", translations=["ES", "FR", "DE", "JA"])
print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "word": "ephemeral",
  "part_of_speech": "adjective",
  "ipa": "/ɪˈfɛmərəl/",
  "definition": "Lasting for a very short time; transitory or fleeting. Something that exists briefly and then disappears.",
  "etymology": "From Greek 'ephemeros,' meaning 'lasting only a day,' from 'epi-' (upon) + 'hemera' (day). Originally described insects that live for only one day, later extended to anything short-lived.",
  "mnemonic": "Think 'a-feminine-al' - like how some people think feminine beauty is fleeting and temporary. But better yet, think of a mayfly that lives for just one day - ephemeral things are here today, gone today!",
  "image_url": "/media/images/words/ephemeral.png",
  "translations": {
    "de": {
      "word": "flüchtig",
      "pronunciation": "/ˈflyːçtɪç/"
    },
    "es": {
      "word": "efímero",
      "pronunciation": "eˈfimɛɾo"
    },
    "fr": {
      "word": "éphémère",
      "pronunciation": "/e.fɛ.mɛʁ/"
    },
    "ja": {
      "word": "エフェメラル",
      "pronunciati

Notice the rich shape: definition, IPA, etymology, mnemonic, translations with pronunciations, and a graph of related words. This is exactly the kind of structured context Claude can reason over.


## Step 3: Define the `lookup_word` tool for Claude

Now we wrap `kelly_lookup` as a Claude tool. The tool description and schema are what Claude reads to decide *when* and *how* to call the tool, so we keep them clear and specific.


In [4]:
tools = [
    {
        "name": "lookup_word",
        "description": (
            "Look up a single English word in the Kelly Intelligence vocabulary "
            "database. Returns the part of speech, IPA pronunciation, definition, "
            "etymology, a mnemonic, translations into the requested languages, and "
            "a list of related words. Use this whenever you need authoritative "
            "vocabulary information."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "word": {
                    "type": "string",
                    "description": "The English headword to look up. Use the base form (e.g. 'run', not 'running').",
                },
                "translations": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": (
                        "Optional list of ISO language codes for translations. "
                        "Supported: ES, FR, DE, IT, PT, JA, ZH, KO, AR, RU, HI, TR, "
                        "PL, NL, SV, VI, ID, TH, HE, EL, CS, DA, FI, NO, RO, HU, UK, "
                        "TA, BN, MS, TL, FA, UR, MY, KM, SW, AM, YO, ZU, TE, MR, PA, "
                        "CA, HA, IG, GU, KK. Max 10."
                    ),
                },
            },
            "required": ["word"],
        },
    }
]

## Step 4: Run Claude's tool-use loop

Here's the simplest possible loop: send a user prompt, let Claude decide whether to call the tool, run the tool, send the result back, and let Claude answer.


In [5]:
def process_tool_call(tool_name: str, tool_input: dict) -> str:
    if tool_name == "lookup_word":
        result = kelly_lookup(
            word=tool_input["word"],
            translations=tool_input.get("translations"),
        )
        return json.dumps(result, ensure_ascii=False)
    raise ValueError(f"Unknown tool: {tool_name}")


def chat_with_kelly_tutor(user_message: str, system: str | None = None) -> str:
    """Chat with Claude, giving it access to the Kelly vocabulary tool."""
    print(f"\n{'=' * 60}\nUser: {user_message}\n{'=' * 60}")

    messages = [{"role": "user", "content": user_message}]
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 1024,
        "tools": tools,
        "messages": messages,
    }
    if system:
        kwargs["system"] = system

    response = client.messages.create(**kwargs)

    # Tool-use loop: keep going until Claude stops asking for tools.
    # Claude may emit multiple parallel tool_use blocks per turn — handle all of them.
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"[tool] {block.name}({json.dumps(block.input, ensure_ascii=False)})")
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": process_tool_call(block.name, block.input),
                    }
                )

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        kwargs["messages"] = messages
        response = client.messages.create(**kwargs)

    # Final text answer
    final = "".join(b.text for b in response.content if b.type == "text")
    print(f"\nClaude:\n{final}")
    return final

## Step 5: Try it out

Let's ask Claude to teach us a word. Notice that we never tell it *which* word to look up — it figures that out from the prompt.


In [6]:
_ = chat_with_kelly_tutor(
    "What does 'serendipity' mean? Give me the etymology and one example sentence.",
)


User: What does 'serendipity' mean? Give me the etymology and one example sentence.


[tool] lookup_word({"word": "serendipity"})



Claude:
**Serendipity** (noun) means **the occurrence of finding pleasant or valuable things by fortunate accident; a happy chance discovery.**

**Etymology:**
The word was coined by Horace Walpole in 1754, inspired by the Persian fairy tale "The Three Princes of Serendip" (the old name for Sri Lanka). In the tale, the heroes constantly made valuable discoveries through accidental encounters and clever thinking.

**Example sentence:**
"Finding that old photograph of my grandparents in the dusty attic was pure serendipity, reminding me of family stories I thought were lost forever."


Now let's give Claude a tutor persona via the system prompt and ask for a richer lesson with translations.


In [7]:
TUTOR_SYSTEM = (
    "You are a friendly vocabulary tutor. When a learner asks about a word, "
    "use the lookup_word tool to fetch authoritative data, then teach the word "
    "in 4 short sections: (1) one-line definition, (2) etymology in one sentence, "
    "(3) translations table for the requested languages, (4) one short example "
    "sentence using the word in context. Keep the whole response under 200 words."
)

_ = chat_with_kelly_tutor(
    "Teach me the word 'ubiquitous'. I'm learning Spanish and Japanese.",
    system=TUTOR_SYSTEM,
)


User: Teach me the word 'ubiquitous'. I'm learning Spanish and Japanese.


[tool] lookup_word({"word": "ubiquitous", "translations": ["ES", "JA"]})



Claude:
## **Ubiquitous** 📍

**Definition:** Something present or seeming to be everywhere at once; extremely common and hard to avoid.

**Etymology:** From Latin *ubique* ("everywhere") + the suffix *-ous* ("full of"), entering English in the 19th century.

**Translations:**
| Language | Translation |
|----------|-------------|
| Spanish | Ubicuo/ubicua |
| Japanese | 至る所にある (itaru tokoro ni aru) |

**Example:** *Smartphones have become ubiquitous in modern society—you rarely see someone without one.*

**Related words:** omnipresent, pervasive, widespread, common, prevalent


## Step 6: Multi-word lessons

Claude can call the tool more than once in a single conversation. Let's ask for a comparison between two related words.


In [8]:
_ = chat_with_kelly_tutor(
    "What's the difference between 'transient' and 'ephemeral'? "
    "Use the lookup tool for both, then explain the nuance in 3 sentences.",
    system=TUTOR_SYSTEM,
)


User: What's the difference between 'transient' and 'ephemeral'? Use the lookup tool for both, then explain the nuance in 3 sentences.


[tool] lookup_word({"word": "transient"})


[tool] lookup_word({"word": "ephemeral"})



Claude:
Great question! Here's the nuance:

**Transient** emphasizes something *passing through* temporarily—it's brief because it's in motion or transition (from Latin "transire," to go across). Think of a transient visitor who stays for a few days but isn't meant to settle.

**Ephemeral** emphasizes something inherently *short-lived by nature*—it's brief because that's simply its lifespan (from Greek "ephemeros," lasting only a day). Think of a mayfly or a fleeting moment of beauty that exists for its brief moment.

**In short**: Transient = temporarily here (could stay longer in theory); Ephemeral = naturally short-lived (destined to be brief). Both are temporary, but ephemeral stresses the *essence* of brevity, while transient stresses *passing through*.


## Where to go next

You now have a working vocabulary tutor that runs on Claude + a public vocabulary API. A few directions to explore:

- **Spaced-repetition flashcards** — pass Claude a list of words a learner is studying and have it generate flashcards with example sentences in the target language.
- **Reading-level analyzer** — give Claude a paragraph and have it look up every advanced word, producing a glossary for younger readers.
- **Etymology agent** — chain `lookup_word` calls along the related-word graph to walk a learner through the historical roots of a concept.

For more on Kelly Intelligence (the OpenAI-compatible Claude proxy and the broader Lesson of the Day developer platform), see [api.thedailylesson.com](https://api.thedailylesson.com).
